# OpenThreat NLP: retrain the relation model only (~10-15 min)

Same as the full notebook but only the relation-extraction part. It now also learns
**"no relation"** (NONE) from generated negative pairs. It uploads to the same Hugging Face repo,
so nothing else needs to change.

**Before you start:** `Runtime â†’ Change runtime type â†’ T4 GPU`, set `HF_USER` in cell 3, then
`Runtime â†’ Run all`. At the end it downloads `openthreat_re_results.zip`. **Send that to Claude.**

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then run again."

In [ ]:
# 2. Install libraries (Colab already has torch)
!pip -q install "transformers==4.46.3" "accelerate>=0.34" "datasets>=2.20" matplotlib

In [ ]:
# 3. Settings â€” CHANGE HF_USER to your Hugging Face username
HF_USER = "your-hf-username"   # your Hugging Face username
BASE_MODEL = "ehsanaghaei/SecureBERT"   # RoBERTa pre-trained on cybersecurity text; fallback: "roberta-base"
NER_REPO = f"{HF_USER}/openthreat-ner"
RE_REPO = f"{HF_USER}/openthreat-re"
assert HF_USER != "your-hf-username", "Set HF_USER first"

from huggingface_hub import login, whoami, create_repo, upload_file
login()   # log in (or paste a WRITE token) when asked
print("Logged in as:", whoami()["name"])
# Check we can WRITE to both repos now, not after 40 minutes of training
for repo in (NER_REPO, RE_REPO):
    create_repo(repo, private=False, exist_ok=True)
    upload_file(path_or_fileobj=b"OpenThreat NLP model (training in progress)", path_in_repo="README.md", repo_id=repo)
    print("Write access OK:", "https://huggingface.co/" + repo)

In [ ]:
%%writefile common.py
"""Shared helpers for training and evaluation: seeding, span-level NER metrics, confusion matrices,
and the experiment log (dataset version, model, hyper-parameters, seed, metrics) required for
reproducibility.
"""
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

# Our v1 taxonomy (must match docs/contracts)
ENTITY_TYPES = ["THREAT_ACTOR", "MALWARE", "TOOL", "VULNERABILITY", "TECHNIQUE", "ORGANIZATION", "OBSERVABLE"]
NER_LABELS = ["O"] + [f"{p}-{t}" for t in ENTITY_TYPES for p in ("B", "I")]
RELATION_TYPES = ["USES", "DEPLOYS", "EXPLOITS", "TARGETS", "ATTRIBUTED_TO"]
RE_LABELS = ["NONE", *RELATION_TYPES, "OTHER"]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ---------------------------------------------------------------- span metrics (entity level)
def bio_to_spans(tags: list[str]) -> set[tuple[int, int, str]]:
    """BIO tags -> {(start, end_exclusive, type)}. A stray I- starts a new span (lenient, like seqeval)."""
    spans, start, etype = set(), None, None
    for i, tag in enumerate([*tags, "O"]):
        if tag == "O" or tag.startswith("B-") or (tag.startswith("I-") and tag[2:] != etype):
            if start is not None:
                spans.add((start, i, etype))
                start, etype = None, None
            if tag != "O":
                start, etype = i, tag[2:]
    return spans


def span_prf(gold: list[list[str]], pred: list[list[str]], types: list[str] = ENTITY_TYPES) -> dict:
    """Exact-match entity-level precision / recall / F1, per type plus micro and macro averages."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(gold, pred):
        gs, ps = bio_to_spans(g), bio_to_spans(p)
        for s in ps & gs:
            tp[s[2]] += 1
        for s in ps - gs:
            fp[s[2]] += 1
        for s in gs - ps:
            fn[s[2]] += 1
    return prf_table(tp, fp, fn, types)


def prf_table(tp: Counter, fp: Counter, fn: Counter, types: list[str]) -> dict:
    def prf(t, f_p, f_n):
        p = t / (t + f_p) if t + f_p else 0.0
        r = t / (t + f_n) if t + f_n else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4), "support": t + f_n}

    out = {t: prf(tp[t], fp[t], fn[t]) for t in types}
    out["micro"] = prf(sum(tp[t] for t in types), sum(fp[t] for t in types), sum(fn[t] for t in types))
    present = [t for t in types if out[t]["support"] > 0]
    out["macro"] = {
        k: round(float(np.mean([out[t][k] for t in present])), 4) if present else 0.0
        for k in ("precision", "recall", "f1")
    }
    out["macro"]["support"] = sum(out[t]["support"] for t in present)
    return out


def classification_prf(gold: list[str], pred: list[str], labels: list[str], positive: list[str]) -> dict:
    """Per-class P/R/F1 for single-label classification; micro/macro computed over `positive` classes only."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(gold, pred):
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1
    table = prf_table(tp, fp, fn, positive)
    for lab in labels:
        if lab not in table:
            table[lab] = prf_table(tp, fp, fn, [lab])[lab]
    table["accuracy"] = round(sum(tp.values()) / max(1, len(gold)), 4)
    return table


def confusion(gold: list[str], pred: list[str], labels: list[str]) -> list[list[int]]:
    idx = {l: i for i, l in enumerate(labels)}
    m = np.zeros((len(labels), len(labels)), dtype=int)
    for g, p in zip(gold, pred):
        if g in idx and p in idx:
            m[idx[g], idx[p]] += 1
    return m.tolist()


def plot_confusion(matrix: list[list[int]], labels: list[str], title: str, path: str | Path, normalize: bool = True) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    m = np.array(matrix, dtype=float)
    shown = m / np.maximum(m.sum(axis=1, keepdims=True), 1) if normalize else m
    fig, ax = plt.subplots(figsize=(1.1 * len(labels) + 2, 1.0 * len(labels) + 1.5))
    im = ax.imshow(shown, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Gold")
    ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = f"{shown[i, j]:.2f}" if normalize else f"{int(m[i, j])}"
            ax.text(j, i, val, ha="center", va="center", fontsize=8,
                    color="white" if shown[i, j] > (0.5 if normalize else shown.max() / 2) else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------- experiment log
def write_experiment_log(out_dir: str | Path, name: str, record: dict) -> Path:
    """Append-only reproducibility log: one JSON file per run, plus a running experiments.jsonl."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    record = {
        "experiment": name,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "python": platform.python_version(),
        **_lib_versions(),
        **record,
    }
    path = out_dir / f"{name}.json"
    path.write_text(json.dumps(record, indent=2))
    with open(out_dir / "experiments.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")
    return path


def _lib_versions() -> dict:
    out = {}
    for lib in ("torch", "transformers", "numpy"):
        try:
            out[f"{lib}_version"] = __import__(lib).__version__
        except ImportError:
            pass
    try:
        import torch

        out["device"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    except ImportError:
        pass
    return out

In [ ]:
%%writefile data_re.py
"""AZERG (QCRI/AZERG-Dataset, Apache-2.0) -> relation classification examples with our labels.

AZERG is stored as instruction/answer pairs for four tasks (T1 entity detection, T2 typing,
T3 "are these two entities related? YES/NO", T4 "which relation label?"). We use:
  * T4 rows -> the STIX label, mapped to our 5 relation types (or OTHER)
  * T3 rows answered NO, and T4 "is not related to" -> NONE (negative examples)
  * AZERG has (almost) no explicit negatives, so more NONE examples are generated from T1 entity
    lists: entity pairs in the same passage that are not annotated as related
Each example is the passage with the two entities wrapped in markers:
  "[E1] APT16 [/E1] ... [E2] ELMER [/E2] ..."  (E1 = source/subject, E2 = target/object)
"""
from __future__ import annotations

import json
import re
from collections import Counter
from dataclasses import dataclass

STIX_TO_OURS = {
    "uses": "USES",
    "drops": "DEPLOYS",
    "downloads": "DEPLOYS",
    "delivers": "DEPLOYS",
    "installs": "DEPLOYS",
    "exploits": "EXPLOITS",
    "targets": "TARGETS",
    "compromises": "TARGETS",
    "attributed-to": "ATTRIBUTED_TO",
    "authored-by": "ATTRIBUTED_TO",
    "is not related to": "NONE",
}
DROP_LABELS = {"not sure"}

MARKERS = ["[E1]", "[/E1]", "[E2]", "[/E2]"]

_INPUT = re.compile(
    r"###\s*Source Entity:\s*(?P<src>.*?)\s*###\s*Target Entity:\s*(?P<tgt>.*?)\s*###.*?###\s*Text Passage:\s*(?P<text>.*)",
    re.S,
)
_LABEL = re.compile(r"<label>\s*(.*?)\s*</label>", re.S)
_RELATED = re.compile(r"<related>\s*(YES|NO)\s*</related>", re.S | re.I)


@dataclass
class REExample:
    text: str  # passage with markers
    subject: str
    object: str
    label: str  # one of common.RE_LABELS
    stix_label: str


def map_label(stix: str) -> str:
    stix = stix.strip().lower()
    return STIX_TO_OURS.get(stix, "OTHER")


def _find(text: str, name: str, taken: tuple[int, int] | None = None) -> tuple[int, int] | None:
    for flags in (0, re.I):
        for m in re.finditer(r"(?<![\w-])" + re.escape(name) + r"(?![\w-])", text, flags):
            if taken and m.start() < taken[1] and taken[0] < m.end():
                continue
            return m.start(), m.end()
    return None


def mark(text: str, subj: str, obj: str) -> str | None:
    """Wrap the first mention of subject and object in markers. None if either is missing."""
    s = _find(text, subj)
    if s is None:
        return None
    o = _find(text, obj, taken=s)
    if o is None:
        return None
    spans = sorted([(s, "[E1]", "[/E1]"), (o, "[E2]", "[/E2]")], key=lambda x: x[0][0], reverse=True)
    for (a, b), open_, close in spans:
        text = text[:a] + f"{open_} {text[a:b]} {close}" + text[b:]
    return text


_PASSAGE = re.compile(r"###\s*Text Passage:\s*(?P<text>.*)", re.S)
_ENTITIES = re.compile(r"<entities>\s*(.*?)\s*</entities>", re.S)


def parse_rows(rows, neg_ratio: float = 1.0, seed: int = 42) -> tuple[list[REExample], Counter]:
    """rows: iterable of dicts with instruction/input/output.

    Positives come from T4 (relation label) rows. AZERG has no explicit negative rows, so negatives
    are generated the standard way: every ordered pair of entities listed for the same passage (T1
    rows) that is not annotated as related becomes NONE, including the reversed direction of a
    related pair. Negatives are sampled to `neg_ratio` x the number of positives.
    """
    import random

    examples: dict[tuple[str, str, str], REExample] = {}
    related_pairs: set[tuple[str, str, str]] = set()  # any annotated pair, incl. T3 YES without a label
    passage_entities: dict[str, list[str]] = {}
    skipped = Counter()
    for row in rows:
        instr, inp, outp = row["instruction"], row["input"] or "", row["output"] or ""
        m = _INPUT.search(inp)
        if not m:
            pm, em = _PASSAGE.search(inp), _ENTITIES.search(outp)
            if pm and em and "extract all" in instr.lower():  # T1: entity list for a passage
                ents = [e.strip() for e in em.group(1).split("|") if e.strip()]
                passage_entities[pm.group("text").strip()] = ents
            continue
        src, tgt, passage = m.group("src").strip(), m.group("tgt").strip(), m.group("text").strip()
        if "label of the relationship" in instr:
            lm = _LABEL.search(outp)
            if not lm:
                skipped["no_label"] += 1
                continue
            stix = lm.group(1).strip().lower()
            related_pairs.add((src, tgt, passage))
            if stix in DROP_LABELS:
                skipped["not_sure"] += 1
                continue
            label = map_label(stix)
        elif "semantically related" in instr:
            rm = _RELATED.search(outp)
            if rm and rm.group(1).upper() == "YES":
                related_pairs.add((src, tgt, passage))
                continue  # the label comes from the matching T4 row
            if not rm:
                continue
            stix, label = "no", "NONE"
        else:
            continue
        marked = mark(passage, src, tgt)
        if marked is None:
            skipped["entity_not_in_passage"] += 1
            continue
        key = (src, tgt, passage)
        if key not in examples or examples[key].label == "NONE":
            examples[key] = REExample(marked, src, tgt, label, stix)

    # ---- generated negatives
    n_pos = sum(1 for e in examples.values() if e.label != "NONE")
    candidates = []
    for passage, ents in passage_entities.items():
        uniq = list(dict.fromkeys(ents))
        for a in uniq:
            for b in uniq:
                if a == b or a.casefold() in b.casefold() or b.casefold() in a.casefold():
                    continue
                if (a, b, passage) in related_pairs or (a, b, passage) in examples:
                    continue
                candidates.append((a, b, passage))
    rng = random.Random(seed)
    rng.shuffle(candidates)
    added = 0
    for a, b, passage in candidates:
        if added >= int(neg_ratio * n_pos):
            break
        marked = mark(passage, a, b)
        if marked is None:
            continue
        examples[(a, b, passage)] = REExample(marked, a, b, "NONE", "generated-negative")
        added += 1
    skipped["generated_negatives"] = added
    skipped["negative_candidates"] = len(candidates)
    return list(examples.values()), skipped


def load_azerg(split: str, local_jsonl: str | None = None) -> tuple[list[REExample], Counter]:
    if local_jsonl:
        rows = [json.loads(l) for l in open(local_jsonl, encoding="utf-8") if l.strip()]
        rows = [r for r in rows if r.get("split", split) == split]
    else:
        from datasets import load_dataset

        rows = load_dataset("QCRI/AZERG-Dataset", split=split)
    return parse_rows(rows)

In [ ]:
%%writefile train_re.py
"""Fine-tune a relation classifier (entity markers + sequence classification) on AZERG.

    python train_re.py --model ehsanaghaei/SecureBERT --out runs/re --push_repo <hf-user>/openthreat-re

Labels: NONE, USES, DEPLOYS, EXPLOITS, TARGETS, ATTRIBUTED_TO, OTHER (see data_re.py).
Reported metrics: per-class P/R/F1, micro/macro over the 5 relation types, confusion matrix.
"""
from __future__ import annotations

import argparse
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np

from common import (RE_LABELS, RELATION_TYPES, classification_prf, confusion, plot_confusion, set_seed,
                    write_experiment_log)
from data_re import MARKERS, load_azerg


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="ehsanaghaei/SecureBERT")
    ap.add_argument("--out", default="runs/re")
    ap.add_argument("--local_jsonl", default=None, help="AZERG rows as JSONL (for offline tests)")
    ap.add_argument("--epochs", type=float, default=4)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--max_len", type=int, default=256)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--push_repo", default="")
    args = ap.parse_args()

    import torch
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer,
                              TrainingArguments)

    set_seed(args.seed)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    train_all, skipped_train = load_azerg("train", args.local_jsonl)
    test, skipped_test = load_azerg("test", args.local_jsonl)
    rng = random.Random(args.seed)
    rng.shuffle(train_all)
    n_dev = max(1, int(0.1 * len(train_all)))
    dev, train = train_all[:n_dev], train_all[n_dev:]
    if args.limit:
        train, dev, test = train[: args.limit], dev[: max(20, args.limit // 5)], test[: max(20, args.limit // 5)]
    data_stats = {name: dict(Counter(e.label for e in items)) for name, items in
                  {"train": train, "dev": dev, "test": test}.items()}
    data_stats["skipped"] = {"train": dict(skipped_train), "test": dict(skipped_test)}
    print(json.dumps(data_stats, indent=1))

    label2id = {l: i for i, l in enumerate(RE_LABELS)}
    id2label = {i: l for l, i in label2id.items()}
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    tokenizer.add_special_tokens({"additional_special_tokens": MARKERS})
    model = AutoModelForSequenceClassification.from_pretrained(
        args.model, num_labels=len(RE_LABELS), id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True
    )
    model.resize_token_embeddings(len(tokenizer))

    def encode(items):
        enc = []
        for e in items:
            x = tokenizer(e.text, truncation=True, max_length=args.max_len)
            x["labels"] = label2id[e.label]
            enc.append(dict(x))
        return enc

    train_ds, dev_ds, test_ds = encode(train), encode(dev), encode(test)

    # class weights (sqrt of inverse frequency) so rare relations such as EXPLOITS are not ignored
    counts = Counter(e.label for e in train)
    weights = torch.tensor([(len(train) / max(1, counts.get(l, 0))) ** 0.5 if counts.get(l) else 0.0
                            for l in RE_LABELS], dtype=torch.float)
    weights = weights / weights[weights > 0].mean()

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weights.to(outputs.logits.device))
            return (loss, outputs) if return_outputs else loss

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        gold = [id2label[int(i)] for i in labels]
        pred = [id2label[int(i)] for i in np.argmax(logits, axis=-1)]
        m = classification_prf(gold, pred, RE_LABELS, RELATION_TYPES)
        return {"macro_f1": m["macro"]["f1"], "micro_f1": m["micro"]["f1"], "accuracy": m["accuracy"]}

    targs = TrainingArguments(
        output_dir=str(out / "checkpoints"),
        num_train_epochs=args.epochs,
        learning_rate=args.lr,
        per_device_train_batch_size=args.batch,
        per_device_eval_batch_size=args.batch * 2,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to=[],
        seed=args.seed,
    )
    trainer = WeightedTrainer(model=model, args=targs, train_dataset=train_ds, eval_dataset=dev_ds,
                              data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
    trainer.train()

    pred_out = trainer.predict(test_ds)
    gold = [id2label[int(i)] for i in pred_out.label_ids]
    pred = [id2label[int(i)] for i in np.argmax(pred_out.predictions, axis=-1)]
    metrics = classification_prf(gold, pred, RE_LABELS, RELATION_TYPES)
    cm = confusion(gold, pred, RE_LABELS)
    plot_confusion(cm, RE_LABELS, "Relation extraction confusion (AZERG test, row-normalised)", out / "confusion_re.png")
    (out / "metrics_re.json").write_text(json.dumps(
        {"test": metrics, "confusion_labels": RE_LABELS, "confusion": cm, "data": data_stats}, indent=2))
    # keep test predictions for error analysis in the report
    with open(out / "predictions_re_test.jsonl", "w") as f:
        for e, p in zip(test, pred):
            f.write(json.dumps({"text": e.text, "gold": e.label, "pred": p, "stix": e.stix_label}) + "\n")

    trainer.save_model(str(out / "model"))
    tokenizer.save_pretrained(str(out / "model"))
    write_experiment_log(out.parent / "experiments", "re_" + Path(args.model).name, {
        "task": "RE",
        "dataset": "QCRI/AZERG-Dataset (T3 negatives + T4 labels)",
        "label_mapping": "training/data_re.py:STIX_TO_OURS",
        "split": "AZERG train (90% train / 10% dev, seed-shuffled) and AZERG test",
        "base_model": args.model,
        "hyperparameters": {k: getattr(args, k) for k in ("epochs", "lr", "batch", "max_len", "limit")},
        "class_weights": dict(zip(RE_LABELS, [round(float(w), 3) for w in weights])),
        "seed": args.seed,
        "metrics": {"test_micro": metrics["micro"], "test_macro": metrics["macro"], "accuracy": metrics["accuracy"],
                    "per_type_f1": {t: metrics[t]["f1"] for t in RELATION_TYPES}},
        "hf_repo": args.push_repo or None,
    })
    print(json.dumps({"micro": metrics["micro"], "macro": metrics["macro"]}, indent=2))

    if args.push_repo:
        model.push_to_hub(args.push_repo, private=False)
        tokenizer.push_to_hub(args.push_repo)
        print("pushed to https://huggingface.co/" + args.push_repo)


if __name__ == "__main__":
    main()

In [ ]:
# 6. Check the base model downloads; fall back to roberta-base if it doesn't
from transformers import AutoTokenizer
try:
    AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)
except Exception as e:
    print("Could not load", BASE_MODEL, "->", e, "\nFalling back to roberta-base")
    BASE_MODEL = "roberta-base"
print("Base model:", BASE_MODEL)

In [ ]:
# 8. Train + evaluate relation extraction (~10-15 min)
import subprocess, sys
subprocess.run([sys.executable, "train_re.py", "--model", BASE_MODEL, "--out", "runs/re",
                "--push_repo", RE_REPO], check=True)  # stops on any error

In [ ]:
# Results
import json
from IPython.display import Image, display
m = json.load(open("runs/re/metrics_re.json"))
re_ = m["test"]
print("Test label counts:", m["data"]["test"])
print("RE test  micro F1:", re_["micro"]["f1"], " macro F1:", re_["macro"]["f1"], " accuracy:", re_["accuracy"])
for t in ["NONE","USES","DEPLOYS","EXPLOITS","TARGETS","ATTRIBUTED_TO","OTHER"]:
    print(f"  {t:<14} P={re_[t]['precision']:.3f} R={re_[t]['recall']:.3f} F1={re_[t]['f1']:.3f} (n={re_[t]['support']})")
display(Image("runs/re/confusion_re.png"))

In [ ]:
# Verify the model is really on Hugging Face
from huggingface_hub import HfApi
files = [f.rfilename for f in HfApi().model_info(RE_REPO).siblings]
assert any(f.endswith((".safetensors", ".bin")) for f in files), f"weights missing! {files}"
print("VERIFIED on Hugging Face:", "https://huggingface.co/" + RE_REPO, "|", len(files), "files")

In [ ]:
# Download the results zip -> send it to Claude
!cd runs && zip -q -r ../openthreat_re_results.zip re/metrics_re.json re/confusion_re.png re/predictions_re_test.jsonl experiments
from google.colab import files
files.download("openthreat_re_results.zip")
print("Model: https://huggingface.co/" + RE_REPO)